# Durable Task Scheduler Orchestrates Sandbox Jobs

Run Durable Task Scheduler (DTS) from this notebook process while the actual workload runs inside Azure Container Apps sandboxes.

## Prerequisites
- Azure CLI: `az login`
- GitHub CLI for the published wheel flow: `gh auth status`
- Install the sandbox SDK wheels from a GitHub Release for **this repo**, then install DTS separately into this notebook kernel:

  ```powershell
  $wheelDir = Join-Path (Get-Location) '.artifacts\release-wheels'
  New-Item -ItemType Directory -Force -Path $wheelDir | Out-Null

  gh release download <tag> --repo Azure-Samples/azure-container-apps-sandboxes --pattern 'azure_sandbox-*.whl' --dir $wheelDir
  gh release download <tag> --repo Azure-Samples/azure-container-apps-sandboxes --pattern 'azure_mgmt_sandbox-*.whl' --dir $wheelDir

  $sdkWheels = Get-ChildItem "$wheelDir\azure_sandbox-*.whl", "$wheelDir\azure_mgmt_sandbox-*.whl" | ForEach-Object FullName
  python -m pip install @sdkWheels
  python -m pip install durabletask-azuremanaged
  ```

- If `vendor\wheels` exists locally, `main.py` can load those sandbox wheels at runtime as a validation or fallback path.
- The notebook provisions DTS with the official `az durabletask` CLI extension.
- The Durable Task worker stays **outside** every sandbox by default.


### 0. Initialize variables

Names are derived from the lab folder and your Azure subscription so you can reuse the lab without hardcoding subscription-specific values.
Set cleanup flags only when you are ready for destructive actions.


In [ ]:
import importlib
import json
import logging
import sys
from pathlib import Path

logging.basicConfig(level=logging.INFO, format="%(message)s")

lab_path = Path(globals().get("__vsc_ipynb_file__", Path.cwd() / "__file__")).resolve().parent
if str(lab_path) not in sys.path:
    sys.path.insert(0, str(lab_path))

import main as main_module
main_module = importlib.reload(main_module)

LabConfig = main_module.LabConfig
build_default_names = main_module.build_default_names
cleanup_sandboxes_by_id = main_module.cleanup_sandboxes_by_id
create_runtime = main_module.create_runtime
delete_dts_resources = main_module.delete_dts_resources
delete_resource_group_resources = main_module.delete_resource_group_resources
delete_sandbox_group_resources = main_module.delete_sandbox_group_resources
extract_sandbox_ids = main_module.extract_sandbox_ids
load_account = main_module.load_account
provision_dts_resources = main_module.provision_dts_resources
run_fan_out_workflow = main_module.run_fan_out_workflow
run_single_workflow = main_module.run_single_workflow
start_runtime = main_module.start_runtime
stop_runtime = main_module.stop_runtime

existing_runtime = globals().get("runtime")
if existing_runtime is not None:
    stop_runtime(existing_runtime)
    runtime = None

account = load_account()
lab_name = lab_path.name
defaults = build_default_names(lab_name, account["id"])

resource_group_name = defaults["resource_group"]
resource_group_location = "westus2"
scheduler_name = defaults["scheduler_name"]
task_hub_name = defaults["task_hub"]
sandbox_group_name = defaults["sandbox_group"]
scheduler_sku = "Consumption"
scheduler_capacity = 1
scheduler_ip_allowlist = "[0.0.0.0/0]"

assign_current_user_role = True
cleanup_sandboxes = True
stop_and_resume = True
fan_out_width = 5
delete_sandbox_group_when_done = True
delete_scheduler_resources = True
delete_resource_group = True

print(f"Lab:            {lab_name}")
print(f"User:           {account['user']['name']}")
print(f"Subscription:   {account['name']} ({account['id']})")
print(f"Resource Group: {resource_group_name}")
print(f"Location:       {resource_group_location}")
print(f"Scheduler:      {scheduler_name}")
print(f"Task Hub:       {task_hub_name}")
print(f"Sandbox Group:  {sandbox_group_name}")


### 1. Provision Durable Task Scheduler resources and the task hub

This cell ensures the resource group, scheduler, and task hub all exist together. Later cells reuse the resulting configuration.


In [ ]:
config = LabConfig(
    lab_name=lab_name,
    subscription_id=account["id"],
    resource_group=resource_group_name,
    location=resource_group_location,
    scheduler_name=scheduler_name,
    task_hub=task_hub_name,
    sandbox_group=sandbox_group_name,
    scheduler_sku=scheduler_sku,
    scheduler_capacity=scheduler_capacity,
    ip_allowlist=scheduler_ip_allowlist,
)

provisioned = provision_dts_resources(
    config,
    assign_current_user_role=assign_current_user_role,
    principal_name=account["user"]["name"],
)
print(json.dumps(provisioned, indent=2))


### 2. Start the Durable Task worker

The worker runs **here** in the notebook kernel, not inside any sandbox. Its activities call the sandbox SDK clients to do the actual sandbox work.


In [ ]:
runtime = create_runtime(config, endpoint=provisioned["endpoint"])
start_runtime(runtime)

print(f"Endpoint: {runtime.endpoint}")
print(f"Task hub: {runtime.taskhub}")
print("Worker location: this notebook process")


### 3. Run the primary sandbox workflow

This orchestration sequences sandbox creation, workload staging, execution, stats capture, snapshot creation, and optional stop/resume.


In [ ]:
single_result = run_single_workflow(
    runtime.client,
    config,
    cleanup_sandboxes=cleanup_sandboxes,
    stop_and_resume=stop_and_resume,
)
print(json.dumps(single_result, indent=2))


### 4. Run a small fan-out example

This orchestration starts a few sandbox jobs in parallel so you can see DTS coordinate multiple activities at once.


In [ ]:
fan_out_result = run_fan_out_workflow(
    runtime.client,
    config,
    fan_out_width=fan_out_width,
    cleanup_sandboxes=cleanup_sandboxes,
)
print(json.dumps(fan_out_result, indent=2))


### 5. Inspect the DTS dashboard

Open the dashboard and paste in the endpoint and task hub values shown below.


In [ ]:
print("Dashboard:", provisioned["dashboard_url"])
print("Endpoint:", runtime.endpoint)
print("Task hub:", runtime.taskhub)
print("Primary orchestration instance:", single_result["instance_id"])
print("Fan-out orchestration instance:", fan_out_result["instance_id"])


### 6. Stop the worker and optionally clean up

The worker is always stopped. Resource deletion is controlled by the flags from cell 0.


In [ ]:
cleanup_summary = {}
stop_runtime(runtime)
cleanup_summary["worker"] = "stopped"

if delete_resource_group:
    cleanup_summary["resource_group"] = delete_resource_group_resources(config)
else:
    if delete_sandbox_group_when_done:
        if not cleanup_sandboxes:
            sandbox_ids = extract_sandbox_ids(single_result) + extract_sandbox_ids(fan_out_result)
            if sandbox_ids:
                cleanup_summary["sandboxes"] = cleanup_sandboxes_by_id(config, sandbox_ids)
        cleanup_summary["sandbox_group"] = delete_sandbox_group_resources(config)

    if delete_scheduler_resources:
        cleanup_summary["dts"] = delete_dts_resources(config)

    if not delete_sandbox_group_when_done and not delete_scheduler_resources:
        cleanup_summary["note"] = "No destructive cleanup requested. Resources were left in place."

print(json.dumps(cleanup_summary, indent=2))
